# Step 1 of 3 — Data preparation (Snowflake → local files)

**What this notebook produces**

| File | What it is | Who reads it |
|---|---|---|
| `local_train_data/chunk_XXXX.parquet` | Daily rows per series, `TRAIN_START` → `VAL_END`, ~100 dealers per file | Step 2 (cache build) |
| `local_train_data/_manifest.json` | The settings the chunks were built with | Step 1 itself (refuses to reuse chunks built with other settings) |
| `shared_calendar.parquet` | One row per date: festive flags + calendar features, `TRAIN_START` → `FORECAST_END` | Steps 2 and 3 |
| `category_vocab.json` | For every static column, the list of categories. Position in the list = integer code. Code 0 = unknown. | Steps 2 and 3 |
| `column_roles.json` | Column roles, dates, window lengths | Steps 2 and 3 |

**What changed vs iteration 2**
1. Categories get an explicit, human-readable code book (`category_vocab.json`) instead of a pickled `OrdinalEncoder`.
   The codes are still integers — that is unavoidable, an embedding table is indexed by integers — but Step 2 now
   *proves* the model looks them up in an embedding table instead of multiplying them as numbers.
2. The shared calendar is built from a dedicated query over **all** series, after checking that every festive
   column has one value per date nationwide. Iteration 2 built it from the first rows of chunk 0.
3. Data-quality checks run inside Snowflake **before** anything is downloaded.
4. Chunks carry a manifest; changing dates or columns and re-running cannot silently reuse old chunks.
5. Only history (`TRAIN_START` → `VAL_END`) is downloaded. Forecast-window rows were only ever used for the
   calendar, which now has its own query.

In [ ]:
import os
import sys
import json
import glob
import gc
import hashlib

import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils            # noqa: F401  (kept for parity with earlier iterations)
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType

## Section 0 — Configuration
Every date and path lives here. Nothing below hard-codes a date.

**How the three periods are used**
- `TRAIN` (2023-04-01 → 2026-04-30): every training window's *output* (the days the loss is computed on) lies inside this range.
- `VAL` (2025-05-01 → 2026-08-31): 488 days. Each validation window is 365 input days + 101 output days, so the
  scored outputs fall on **2026-05-01 → 2026-08-31** — after `TRAIN_END`, so no overlap with training targets.
  The first 365 days of `VAL` are only used as encoder input.
- `FORECAST` (2026-09-01 → 2026-12-10): 101 days, so `OUTPUT_CHUNK_LENGTH = 101`.

In [ ]:
TABLE_NAME        = "MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES"
VALID_SERIES_TBL  = "MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING"

local_train_dir = "./local_train_data"
calendar_path   = "./shared_calendar.parquet"
vocab_path      = "./category_vocab.json"
roles_path      = "./column_roles.json"

time_col   = "CAL_DATE"
group_col  = "PARENT_DEALER_CODE_MODEL_FAMILY"
dealer_col = "PARENT_DEALER_CODE"
target_col = "NET_SALES"
FREQ       = "D"
CHUNK_SIZE = 100                      # dealers per parquet file

TRAIN_START    = "2023-04-01"
TRAIN_END      = "2026-04-30"
VAL_START      = "2025-05-01"
VAL_END        = "2026-08-31"
FORECAST_START = "2026-09-01"
FORECAST_END   = "2026-12-10"

INPUT_CHUNK_LENGTH  = 365
OUTPUT_CHUNK_LENGTH = (pd.Timestamp(FORECAST_END) - pd.Timestamp(FORECAST_START)).days + 1   # 101

# columns that are strings but are NOT static covariates
NON_STATIC_STRING_COLS = ["MODEL_FAMILY_CODE", "DAY_OF_THE_WEEK", group_col]
# raw calendar columns: replaced by hand-built features in Section 6
CALENDAR_RAW_COLS = ["YEAR", "MONTH", "DAY_OF_THE_MONTH", "DAY_OF_THE_WEEK"]
# columns dropped before anything else
DROP_COLS = ["FIRST_SALE_DATE", "DATASET_TYPE"]

# --- sanity checks on the configuration itself --------------------------------
_d = {k: pd.Timestamp(v) for k, v in dict(TRAIN_START=TRAIN_START, TRAIN_END=TRAIN_END,
                                           VAL_START=VAL_START, VAL_END=VAL_END,
                                           FORECAST_START=FORECAST_START, FORECAST_END=FORECAST_END).items()}
assert _d["TRAIN_START"] < _d["TRAIN_END"] < _d["VAL_END"] < _d["FORECAST_START"] <= _d["FORECAST_END"]
assert _d["FORECAST_START"] == _d["VAL_END"] + pd.Timedelta(days=1), \
    "History must end the day before the forecast starts."
_val_days = (_d["VAL_END"] - _d["VAL_START"]).days + 1
assert _val_days >= INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH, \
    f"VAL period is {_val_days} days; one validation window needs {INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH}."
_val_scored_start = _d["VAL_START"] + pd.Timedelta(days=INPUT_CHUNK_LENGTH)
print(f"ICL / OCL               : {INPUT_CHUNK_LENGTH} / {OUTPUT_CHUNK_LENGTH}")
print(f"Validation windows      : {_val_days - INPUT_CHUNK_LENGTH - OUTPUT_CHUNK_LENGTH + 1} per series")
print(f"Validation scored dates : {_val_scored_start.date()} -> {_d['VAL_END'].date()}")
print(f"Overlap with training targets: "
      f"{'NONE' if _val_scored_start > _d['TRAIN_END'] else 'YES -- val_loss partly measures memorisation'}")
print("NOTE: the scored validation dates contain no festive season (Navratri/Diwali fall in Oct/Nov).")

## Section 1 — Snowflake session and source query
Only series listed in `VALID_SERIES_FOR_DAILY_MODELLING` are read. Nothing is downloaded yet;
`df` is a lazy Snowpark query.

In [ ]:
session = Session.builder.configs(Snowflake_configuration.ds1_role_json).create()
session.use_database("MOP_DATABASE")
session.use_schema("SOQ")

df = session.sql(f"""
    SELECT * FROM {TABLE_NAME}
    WHERE {group_col} IN (SELECT {group_col} FROM {VALID_SERIES_TBL})
""")

# strip quote characters Snowflake keeps in some column names
for old in df.columns:
    new = old.replace('"', "")
    if new != old:
        df = df.rename(old, new)

df = df.drop(*[c for c in DROP_COLS if c in df.columns])
df = df.with_column(group_col, F.replace(F.col(group_col), F.lit("<>"), F.lit("_")))
print(f"{len(df.columns)} columns after dropping {DROP_COLS}")

## Section 2 — Column roles
- **Static covariates**: every string column except the identifiers. One value per series.
- **Festive covariates**: every remaining non-identifier column (the N/D/C blocks and single-day festivals).

`MODEL_NAME` should already be absent (dropped upstream because it is 1:1 with `MODEL_FAMILY`);
the assertions make that explicit so a schema change cannot quietly bring it back.

In [ ]:
id_cols = [time_col, group_col, target_col]

static_covariates = [
    f.name.replace('"', "") for f in df.schema
    if isinstance(f.datatype, StringType)
    and f.name.replace('"', "") not in NON_STATIC_STRING_COLS
]
all_cols = [c.replace('"', "") for c in df.columns]      # Snowpark quotes names like "N-16"
festive_covariates = [
    c for c in all_cols
    if c not in static_covariates and c not in id_cols
    and c not in CALENDAR_RAW_COLS and c not in NON_STATIC_STRING_COLS
]

assert "MODEL_FAMILY" in static_covariates, "MODEL_FAMILY must be a static covariate."
assert "MODEL_NAME" not in static_covariates, "MODEL_NAME is 1:1 with MODEL_FAMILY and should have been dropped."
assert dealer_col in static_covariates

print("static covariates :", static_covariates)
print(f"festive covariates: {len(festive_covariates)} ->", festive_covariates)

## Section 3 — Data-quality checks (run inside Snowflake)
Each check prints a one-line verdict. A failed check raises with an explanation; fix upstream rather than
patching locally.

| Check | Why it matters |
|---|---|
| Target nulls / negatives | The Negative-Binomial likelihood is undefined below 0 |
| One row per series per day, no gaps | A gap shifts every later day; the model indexes days by position |
| Static columns constant within a series | A series can have only one dealer/family/colour |
| Static columns not null | A null would become its own category |
| Festive columns identical across series on each date | Required for one shared calendar (Section 6) |

In [ ]:
hist = df.filter((F.col(time_col) >= TRAIN_START) & (F.col(time_col) <= VAL_END))

# --- 3a target ---------------------------------------------------------------
r = hist.select(
    F.count(F.lit(1)).alias("N_ROWS"),
    F.sum(F.when(F.col(target_col).is_null(), 1).otherwise(0)).alias("N_NULL"),
    F.sum(F.when(F.col(target_col) < 0, 1).otherwise(0)).alias("N_NEG"),
    F.sum(F.when(F.col(target_col) != F.round(F.col(target_col)), 1).otherwise(0)).alias("N_NON_INTEGER"),
).collect()[0].as_dict()
print("3a target:", r)
assert r["N_NULL"] == 0 and r["N_NEG"] == 0, "Target has nulls or negatives -- fix the source table."
assert r["N_NON_INTEGER"] == 0, "Target has non-integer values; a count likelihood needs integers."

# --- 3b one row per day, no gaps --------------------------------------------
span = (hist.group_by(group_col)
            .agg(F.count(F.lit(1)).alias("N"), F.min(time_col).alias("D0"), F.max(time_col).alias("D1"))
            .with_column("EXPECTED", F.datediff("day", F.col("D0"), F.col("D1")) + 1))
r = span.select(
    F.count(F.lit(1)).alias("N_SERIES"),
    F.sum(F.when(F.col("N") < F.col("EXPECTED"), 1).otherwise(0)).alias("N_WITH_GAPS"),
    F.sum(F.when(F.col("N") > F.col("EXPECTED"), 1).otherwise(0)).alias("N_WITH_DUPLICATE_DAYS"),
    F.sum(F.when(F.col("D1") < F.lit(VAL_END), 1).otherwise(0)).alias("N_ENDING_BEFORE_VAL_END"),
).collect()[0].as_dict()
print("3b continuity:", r)
assert r["N_WITH_GAPS"] == 0 and r["N_WITH_DUPLICATE_DAYS"] == 0, "Series have missing or duplicate days."
if r["N_ENDING_BEFORE_VAL_END"]:
    print(f"   NOTE: {r['N_ENDING_BEFORE_VAL_END']} series stop before {VAL_END}; they cannot be forecast "
          f"from {FORECAST_START} and will be excluded in Step 2.")

# --- 3c / 3d static columns --------------------------------------------------
per_series = hist.group_by(group_col).agg(*[F.count_distinct(F.col(c)).alias(f"S{i}")
                                            for i, c in enumerate(static_covariates)])
mx = per_series.select(*[F.max(F.col(f"S{i}")).alias(f"S{i}")
                         for i in range(len(static_covariates))]).collect()[0].as_dict()
not_constant = [c for i, c in enumerate(static_covariates) if (mx[f"S{i}"] or 0) > 1]
nulls = hist.select(*[F.sum(F.when(F.col(c).is_null(), 1).otherwise(0)).alias(f"S{i}")
                      for i, c in enumerate(static_covariates)]).collect()[0].as_dict()
has_null = [c for i, c in enumerate(static_covariates) if (nulls[f"S{i}"] or 0) > 0]
print("3c static columns varying within a series:", not_constant or "none")
print("3d static columns with nulls             :", has_null or "none")
assert not not_constant, f"These static columns change within a series: {not_constant}"
assert not has_null, f"These static columns contain nulls: {has_null}"

# --- 3e festive columns national ---------------------------------------------
cal_src = df.filter((F.col(time_col) >= TRAIN_START) & (F.col(time_col) <= FORECAST_END))
per_date = cal_src.group_by(time_col).agg(*[F.count_distinct(F.col(c)).alias(f"C{i}")
                                            for i, c in enumerate(festive_covariates)])
mx = per_date.select(*[F.max(F.col(f"C{i}")).alias(f"C{i}")
                       for i in range(len(festive_covariates))]).collect()[0].as_dict()
regional = [c for i, c in enumerate(festive_covariates) if (mx[f"C{i}"] or 0) > 1]
print("3e festive columns that differ across series on the same date:", regional or "none")
assert not regional, (
    f"{regional} vary across series on the same date (regional festivals?). A single shared calendar "
    "would apply one region's dates to everyone. Move these to per-series covariates before continuing.")

## Section 4 — Category vocabularies (the code book for embeddings)
For each static column: the sorted list of distinct values, with `"__UNKNOWN__"` at position 0.
A category's **integer code is its position in this list**. Example: if
`vocab["MODEL_FAMILY"] = ["__UNKNOWN__", "GLAMOUR", "XOOM", ...]` then GLAMOUR → 1, XOOM → 2.

Why reserve 0: at prediction time a category never seen in training (a new dealer) maps to 0,
which is a valid row of the embedding table. The old `unknown_value=-1` is not a valid row and would crash.

The code is only an index into a table of learned vectors. Code 2 is not "twice" code 1. Step 2 has a test
that proves the model treats it that way.

In [ ]:
vocab = {}
for c in static_covariates:
    values = sorted(str(r[0]) for r in hist.select(F.col(c)).distinct().collect())
    vocab[c] = ["__UNKNOWN__"] + values
    print(f"  {c:20s} {len(values):5d} categories   e.g. {values[:3]}")

with open(vocab_path, "w") as f:
    json.dump(vocab, f, indent=1)
print(f"\nVocabulary written -> {vocab_path}")

## Section 5 — Download history in dealer chunks
Rows from `TRAIN_START` to `VAL_END`, ~100 dealers per parquet file. Only the columns Step 2 needs are pulled:
keys, target and static columns. Festive flags come from the shared calendar instead.

**Stale-chunk guard:** `_manifest.json` records the settings. If chunks exist with different settings, the
notebook stops and asks you to delete the folder. It does not mix old and new files.

In [ ]:
chunk_cols = [time_col, group_col, target_col] + static_covariates
settings = dict(table=TABLE_NAME, valid=VALID_SERIES_TBL, start=TRAIN_START, end=VAL_END,
                cols=chunk_cols, chunk_size=CHUNK_SIZE)
fingerprint = hashlib.md5(json.dumps(settings, sort_keys=True).encode()).hexdigest()
manifest_file = os.path.join(local_train_dir, "_manifest.json")

os.makedirs(local_train_dir, exist_ok=True)
existing = glob.glob(os.path.join(local_train_dir, "chunk_*.parquet"))
if existing:
    old = json.load(open(manifest_file)) if os.path.exists(manifest_file) else {}
    if old.get("fingerprint") != fingerprint:
        raise RuntimeError(
            f"{local_train_dir} holds {len(existing)} chunks built with different settings (or no manifest). "
            "Delete the folder and re-run; mixing old and new chunks corrupts the dataset.")
    print(f"Chunks with identical settings already exist ({len(existing)} files) -- reusing them.")
else:
    dealers = sorted(r[0] for r in hist.select(dealer_col).distinct().collect())
    chunks = [dealers[i:i + CHUNK_SIZE] for i in range(0, len(dealers), CHUNK_SIZE)]
    print(f"{len(dealers)} dealers -> {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        out = os.path.join(local_train_dir, f"chunk_{i:04d}.parquet")
        part = hist.select(*chunk_cols).filter(F.col(dealer_col).isin(chunk)).to_pandas()
        part[target_col] = part[target_col].astype(np.float32)
        part.to_parquet(out, index=False)
        print(f"  chunk {i+1}/{len(chunks)}: {len(part):,} rows, {part[group_col].nunique():,} series")
        del part
        gc.collect()
    json.dump({"fingerprint": fingerprint, "settings": settings, "n_chunks": len(chunks)},
              open(manifest_file, "w"), indent=1)

## Section 6 — Shared calendar (festive + calendar features)
**Why one shared calendar:** every festive and calendar feature depends only on the date, not the series.
Section 3e confirmed every series has the same festive value on a given date. So the model gets one
1,350-row table and every series points to the same object. That is ~31K× less RAM than a copy per series,
and training and prediction provably read the same file.

The calendar must reach `FORECAST_END`, because future covariates are needed for every forecast day.

Calendar features (kept from iteration 2):
- `DOW_SIN/COS` — weekday on a circle, so Sunday and Monday are neighbours.
- `IS_MONTH_END`, `IS_MONTH_START`, `DAYS_TO_MONTH_END` — the month-end retail push is a step, not a smooth cycle.
- Month-of-year and day-of-month are deliberately **excluded**: together they pin an exact calendar date and
  let the model copy last year's peak date instead of tracking Diwali (diagnosed in iteration 1).
- `YEAR` is excluded: 2026 values at forecast time would be out of range for anything the model learned.

In [ ]:
cal = (cal_src.group_by(time_col)
              .agg(*[F.max(F.col(c)).alias(f"C{i}") for i, c in enumerate(festive_covariates)])
              .to_pandas())
cal.columns = [time_col] + festive_covariates          # restore real names (aliases avoid quoting issues)
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.sort_values(time_col).reset_index(drop=True)

full = pd.date_range(TRAIN_START, FORECAST_END, freq=FREQ)
missing = full.difference(cal[time_col])
assert len(missing) == 0, (
    f"Calendar is missing {len(missing)} dates (first: {list(missing[:3])}). If the gap is at the end, "
    f"extend {TABLE_NAME} to {FORECAST_END}.")
assert cal[festive_covariates].notna().all().all(), "Festive columns contain nulls."

dt = cal[time_col].dt
cal["DOW_SIN"] = np.sin(2 * np.pi * dt.dayofweek / 7).astype(np.float32)
cal["DOW_COS"] = np.cos(2 * np.pi * dt.dayofweek / 7).astype(np.float32)
cal["IS_MONTH_END"]      = (dt.day >= dt.days_in_month - 2).astype(np.float32)
cal["IS_MONTH_START"]    = (dt.day <= 3).astype(np.float32)
cal["DAYS_TO_MONTH_END"] = ((dt.days_in_month - dt.day) / 31.0).astype(np.float32)
calendar_covariates = ["DOW_SIN", "DOW_COS", "IS_MONTH_END", "IS_MONTH_START", "DAYS_TO_MONTH_END"]

cal[festive_covariates] = cal[festive_covariates].astype(np.float32)
future_covariates = festive_covariates + calendar_covariates
cal[[time_col] + future_covariates].to_parquet(calendar_path, index=False)

fc = cal[(cal[time_col] >= FORECAST_START) & (cal[time_col] <= FORECAST_END)]
print(f"Calendar: {cal[time_col].min().date()} -> {cal[time_col].max().date()} ({len(cal)} days), "
      f"{len(future_covariates)} future covariates")
for anchor in ["N", "D"]:
    if anchor in fc.columns:
        print(f"  '{anchor}' falls on: {[d.date().isoformat() for d in fc.loc[fc[anchor] != 0, time_col]]}")

## Section 7 — Column roles for Steps 2 and 3

In [ ]:
roles = dict(
    time_col=time_col, group_col=group_col, dealer_col=dealer_col, target_col=target_col, freq=FREQ,
    static_covariates=static_covariates, festive_covariates=festive_covariates,
    calendar_covariates=calendar_covariates, future_covariates=future_covariates,
    train_start=TRAIN_START, train_end=TRAIN_END, val_start=VAL_START, val_end=VAL_END,
    forecast_start=FORECAST_START, forecast_end=FORECAST_END,
    input_chunk_length=INPUT_CHUNK_LENGTH, output_chunk_length=OUTPUT_CHUNK_LENGTH,
    vocab_path=vocab_path, calendar_path=calendar_path, chunk_fingerprint=fingerprint,
)
with open(roles_path, "w") as f:
    json.dump(roles, f, indent=2)
print(f"Roles written -> {roles_path}")

session.close()
print("Snowflake session closed.")